# Benchmark analysis (`runs/benchmarks`)

This notebook reads benchmark artifacts under `runs/benchmarks/` and produces **simple tables + graphs**.

It supports:
- **Random chess benchmark suite** from `scripts/chess/benchmark_random.py` (artifacts: `manifest.json`, `summary.json`, `results.jsonl`)
- **Labeled scenario benchmark suite** from `psf benchmark` (artifact: `suite_summary.json`)

Docs/code references:
- `docs/chess.md`
- `docs/benchmarking.md`
- `scripts/chess/benchmark_random.py`
- `src/psf/cli.py` (for `psf benchmark`)


In [ ]:
# Optional (uncomment if you don't have these installed in your Jupyter environment):
# !pip install -q pandas matplotlib

from __future__ import annotations

import json
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'runs').exists():
            return p
    return start


REPO_ROOT = find_repo_root()
BENCH_ROOT = REPO_ROOT / 'runs' / 'benchmarks'
REPO_ROOT, BENCH_ROOT


In [ ]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)


def iter_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def parse_iso(dt: str | None) -> datetime | None:
    if not dt:
        return None
    try:
        return datetime.fromisoformat(dt.replace('Z', '+00:00'))
    except Exception:
        return None


def scan_benchmarks(root: Path) -> list[dict]:
    records: list[dict] = []
    if not root.exists():
        return records

    for d in sorted([p for p in root.iterdir() if p.is_dir()]):
        manifest_path = d / 'manifest.json'
        summary_path = d / 'summary.json'
        suite_summary_path = d / 'suite_summary.json'

        rec: dict = {
            'benchmark_id': d.name,
            'dir': str(d),
            'has_manifest': manifest_path.exists(),
            'has_summary': summary_path.exists(),
            'has_suite_summary': suite_summary_path.exists(),
            'has_positions': (d / 'positions.jsonl').exists(),
            'has_results': (d / 'results.jsonl').exists(),
            'has_log': (d / 'benchmark.log').exists() or (d / 'run.log').exists(),
        }

        if rec['has_summary'] and rec['has_manifest']:
            rec['type'] = 'chess_random_suite'
        elif rec['has_suite_summary']:
            rec['type'] = 'psf_benchmark_suite'
        else:
            rec['type'] = 'unknown_or_incomplete'

        if rec['has_manifest']:
            m = load_json(manifest_path)
            rec['task'] = m.get('task')
            rec['started_at'] = m.get('started_at')
            rec['positions'] = m.get('generated_positions')
            rec['plies'] = m.get('plies')
            rec['seed'] = m.get('seed')
            rec['illegal_ratio'] = m.get('illegal_ratio')
            rec['num_generators'] = len(m.get('generator_systems') or [])
            rec['num_validators'] = len(m.get('validator_systems') or [])

        if rec['has_summary']:
            s = load_json(summary_path)
            rec['finished_at'] = s.get('finished_at')
            rec['generators'] = len((s.get('generators') or {}).keys())
            rec['validators'] = len((s.get('validators') or {}).keys())
            rec['pipeline_enabled'] = bool((s.get('pipeline') or {}).get('enabled'))

        if rec['has_suite_summary']:
            ss = load_json(suite_summary_path)
            rec['created_at'] = ss.get('created_at')
            rec['system_count'] = ss.get('system_count')
            rec['scenario_count'] = len(ss.get('scenarios') or [])

        records.append(rec)

    return records


benchmarks = scan_benchmarks(BENCH_ROOT)
len(benchmarks)


In [ ]:
def show_table(rows: list[dict], *, sort_by: str | None = None, descending: bool = True):
    if not rows:
        return rows
    if pd is None:
        if sort_by:
            rows = sorted(rows, key=lambda r: (r.get(sort_by) is None, r.get(sort_by)), reverse=descending)
        return rows
    df = pd.DataFrame(rows)
    if sort_by and sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=not descending)
    return df


overview = show_table(benchmarks, sort_by='started_at', descending=False)
overview


## What these benchmarks measure

### Random chess benchmark suite (`scripts/chess/benchmark_random.py`)

For each run, the script generates `N` **random chess positions** by playing `--plies` random *legal* plies from the initial position (seeded RNG).

It can evaluate:
- **Validator baseline**: a legality-check system predicts `is_legal` for a candidate move labeled by a Python chess oracle (`python-chess`). Accuracy is reported.
- **Generator baseline**: a move-generation system outputs a move (SAN or UCI). We report:
  - `oracle_parse_rate`: fraction of outputs parseable by the oracle
  - `oracle_legal_rate`: fraction of outputs that are legal moves in the position
- **Validator-on-generator**: a validator judges a generator's move and is compared against the oracle.
- **Generator→validator pipeline**: retry generator with `illegal_moves` until validator accepts (or attempts exhausted).

Primary artifacts:
- `manifest.json`: run configuration + system list
- `summary.json`: aggregate metrics (what we plot below)
- `results.jsonl`: per-position details (used for the deep-dive section)

### Labeled scenario benchmark suite (`psf benchmark`)
Writes `suite_summary.json` with per-system accuracy across labeled scenario inputs.


In [ ]:
def load_random_suite_run(run_dir: Path):
    manifest = load_json(run_dir / 'manifest.json')
    summary = load_json(run_dir / 'summary.json')
    label_by_id: dict[str, str] = {}
    for entry in (manifest.get('generator_systems') or []) + (manifest.get('validator_systems') or []):
        sid = entry.get('id')
        if sid:
            label_by_id[sid] = entry.get('label') or sid
    return manifest, summary, label_by_id


generator_rows: list[dict] = []
for b in benchmarks:
    if b.get('type') != 'chess_random_suite' or not b.get('has_summary'):
        continue
    run_dir = Path(b['dir'])
    manifest, summary, label_by_id = load_random_suite_run(run_dir)
    token_totals = summary.get('token_totals') or {}

    for system_id, stats in (summary.get('generators') or {}).items():
        total = int(stats.get('total') or 0)
        tokens_total = token_totals.get(system_id)
        generator_rows.append(
            {
                'benchmark_id': summary.get('benchmark_id') or run_dir.name,
                'started_at': manifest.get('started_at'),
                'finished_at': summary.get('finished_at'),
                'positions': summary.get('positions'),
                'plies': summary.get('plies'),
                'seed': summary.get('seed'),
                'system_id': system_id,
                'system_label': label_by_id.get(system_id, system_id),
                'oracle_parse_rate': stats.get('oracle_parse_rate'),
                'oracle_legal_rate': stats.get('oracle_legal_rate'),
                'oracle_parseable': stats.get('oracle_parseable'),
                'oracle_legal': stats.get('oracle_legal'),
                'errors': stats.get('errors'),
                'missing_output': stats.get('missing_output'),
                'invalid_move': stats.get('invalid_move'),
                'tokens_total': tokens_total,
                'tokens_per_position': (tokens_total / total) if (tokens_total is not None and total) else None,
            }
        )


show_table(generator_rows, sort_by='oracle_legal_rate')


In [ ]:
def plot_bar_by_benchmark(rows: list[dict], *, metric: str, title: str, percent: bool = True):
    if plt is None:
        print('matplotlib not available; install with: pip install matplotlib')
        return
    by_benchmark: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        by_benchmark[str(r.get('benchmark_id'))].append(r)

    for benchmark_id, items in sorted(by_benchmark.items()):
        items = [i for i in items if i.get(metric) is not None]
        items = sorted(items, key=lambda x: float(x.get(metric) or 0.0), reverse=True)
        if not items:
            continue

        labels = [i.get('system_label') or i.get('system_id') for i in items]
        values = [float(i.get(metric) or 0.0) * (100.0 if percent else 1.0) for i in items]

        fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(labels))))
        ax.barh(labels, values)
        ax.invert_yaxis()
        ax.set_title(f'{title} — {benchmark_id}')
        ax.set_xlabel(f"{metric}{' (%)' if percent else ''}")
        ax.grid(axis='x', alpha=0.2)
        plt.show()


def plot_scatter(rows: list[dict], *, x: str, y: str, title: str, x_label: str, y_label: str):
    if plt is None:
        print('matplotlib not available; install with: pip install matplotlib')
        return
    points = []
    for r in rows:
        xv = r.get(x)
        yv = r.get(y)
        if xv is None or yv is None:
            continue
        points.append((float(xv), float(yv), r.get('system_label') or r.get('system_id')))
    if not points:
        return

    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    labels = [p[2] for p in points]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(xs, ys)
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.grid(alpha=0.2)

    if len(points) <= 10:
        for x0, y0, label in points:
            ax.annotate(label, (x0, y0), textcoords='offset points', xytext=(5, 5), fontsize=8, alpha=0.8)

    plt.show()


# Generator plots
plot_bar_by_benchmark(generator_rows, metric='oracle_legal_rate', title='Generator oracle legal rate', percent=True)
plot_bar_by_benchmark(generator_rows, metric='oracle_parse_rate', title='Generator oracle parse rate', percent=True)
plot_scatter(
    generator_rows,
    x='tokens_per_position',
    y='oracle_legal_rate',
    title='Generator: avg tokens/position vs oracle legal rate',
    x_label='Tokens per position (avg)',
    y_label='Oracle legal rate (fraction)',
)


In [ ]:
validator_rows: list[dict] = []
for b in benchmarks:
    if b.get('type') != 'chess_random_suite' or not b.get('has_summary'):
        continue
    run_dir = Path(b['dir'])
    manifest, summary, label_by_id = load_random_suite_run(run_dir)
    token_totals = summary.get('token_totals') or {}

    for system_id, stats in (summary.get('validators') or {}).items():
        total = int(stats.get('total') or 0)
        tokens_total = token_totals.get(system_id)
        validator_rows.append(
            {
                'benchmark_id': summary.get('benchmark_id') or run_dir.name,
                'started_at': manifest.get('started_at'),
                'finished_at': summary.get('finished_at'),
                'positions': summary.get('positions'),
                'plies': summary.get('plies'),
                'seed': summary.get('seed'),
                'illegal_ratio': summary.get('illegal_ratio'),
                'system_id': system_id,
                'system_label': label_by_id.get(system_id, system_id),
                'accuracy': stats.get('accuracy'),
                'correct': stats.get('correct'),
                'errors': stats.get('errors'),
                'tokens_total': tokens_total,
                'tokens_per_position': (tokens_total / total) if (tokens_total is not None and total) else None,
            }
        )


show_table(validator_rows, sort_by='accuracy')


In [ ]:
# Validator plots
plot_bar_by_benchmark(validator_rows, metric='accuracy', title='Validator baseline accuracy', percent=True)
plot_scatter(
    validator_rows,
    x='tokens_per_position',
    y='accuracy',
    title='Validator: avg tokens/position vs accuracy',
    x_label='Tokens per position (avg)',
    y_label='Accuracy (fraction)',
)


In [ ]:
pipeline_rows: list[dict] = []
for b in benchmarks:
    if b.get('type') != 'chess_random_suite' or not b.get('has_summary'):
        continue
    run_dir = Path(b['dir'])
    manifest, summary, _ = load_random_suite_run(run_dir)
    pipeline = summary.get('pipeline') or {}
    if int(pipeline.get('total') or 0) == 0:
        continue
    pipeline_rows.append(
        {
            'benchmark_id': summary.get('benchmark_id') or run_dir.name,
            'started_at': manifest.get('started_at'),
            'finished_at': summary.get('finished_at'),
            'total': pipeline.get('total'),
            'accept_rate': pipeline.get('accept_rate'),
            'oracle_legal_rate': pipeline.get('oracle_legal_rate'),
            'avg_attempts': pipeline.get('avg_attempts'),
            'false_accepts': pipeline.get('false_accepts'),
            'errors': pipeline.get('errors'),
        }
    )


show_table(pipeline_rows, sort_by='accept_rate')


## Deep dive: per-position results (`results.jsonl`)

Use this section when you want more than aggregates.

- For **validators**, we compute a simple confusion matrix (TP/TN/FP/FN) from `validator_baseline` entries.
- For **generators**, we surface the most common oracle parse errors (when outputs aren't valid SAN/UCI).


In [ ]:
def pick_latest_completed_random_suite(benchmarks: list[dict]) -> Path | None:
    candidates: list[tuple[datetime, Path]] = []
    for b in benchmarks:
        if b.get('type') != 'chess_random_suite':
            continue
        if not b.get('has_summary') or not b.get('has_results'):
            continue
        run_dir = Path(b['dir'])
        try:
            summary = load_json(run_dir / 'summary.json')
        except Exception:
            continue
        finished = parse_iso(summary.get('finished_at'))
        if finished is None:
            finished = datetime(1970, 1, 1, tzinfo=timezone.utc)
        candidates.append((finished, run_dir))
    if not candidates:
        return None
    candidates.sort(key=lambda t: t[0])
    return candidates[-1][1]


def validator_confusion(results_path: Path):
    counts: dict[str, Counter] = defaultdict(Counter)
    for rec in iter_jsonl(results_path):
        vb = rec.get('validator_baseline') or {}
        for system_id, entry in vb.items():
            expected = entry.get('expected_is_legal')
            predicted = entry.get('predicted_is_legal')
            if expected is None:
                continue
            if predicted is None:
                counts[system_id]['invalid'] += 1
            elif bool(expected) and bool(predicted):
                counts[system_id]['tp'] += 1
            elif (not bool(expected)) and (not bool(predicted)):
                counts[system_id]['tn'] += 1
            elif (not bool(expected)) and bool(predicted):
                counts[system_id]['fp'] += 1
            elif bool(expected) and (not bool(predicted)):
                counts[system_id]['fn'] += 1
    rows: list[dict] = []
    for system_id, c in counts.items():
        tp = int(c.get('tp') or 0)
        tn = int(c.get('tn') or 0)
        fp = int(c.get('fp') or 0)
        fn = int(c.get('fn') or 0)
        invalid = int(c.get('invalid') or 0)
        total = tp + tn + fp + fn + invalid
        rows.append(
            {
                'system_id': system_id,
                'total': total,
                'accuracy': (tp + tn) / total if total else 0.0,
                'tp': tp,
                'tn': tn,
                'fp': fp,
                'fn': fn,
                'invalid': invalid,
                'legal_recall': tp / (tp + fn) if (tp + fn) else None,
                'illegal_recall': tn / (tn + fp) if (tn + fp) else None,
            }
        )
    return rows, counts


def generator_parse_errors(results_path: Path):
    errors_by_system: dict[str, Counter] = defaultdict(Counter)
    totals: dict[str, int] = defaultdict(int)
    for rec in iter_jsonl(results_path):
        gen = rec.get('generator') or {}
        for system_id, entry in gen.items():
            totals[system_id] += 1
            err = entry.get('oracle_error')
            if err:
                errors_by_system[system_id][str(err)] += 1
    rows: list[dict] = []
    for system_id, ctr in errors_by_system.items():
        rows.append(
            {
                'system_id': system_id,
                'total_positions': totals.get(system_id, 0),
                'parse_errors': sum(ctr.values()),
                'top_parse_errors': ctr.most_common(5),
            }
        )
    return rows


latest_run = pick_latest_completed_random_suite(benchmarks)
latest_run


In [ ]:
if latest_run is None:
    print('No completed chess_random_suite runs with results.jsonl found.')
else:
    manifest, summary, label_by_id = load_random_suite_run(latest_run)
    results_path = latest_run / 'results.jsonl'
    print('Deep dive run:', latest_run.name)

    # Validator confusion matrix summary
    vc_rows, vc_counts = validator_confusion(results_path)
    if vc_rows:
        vc_df = show_table(
            [
                {
                    **r,
                    'system_label': label_by_id.get(r['system_id'], r['system_id']),
                    'accuracy_%': r['accuracy'] * 100,
                }
                for r in vc_rows
            ],
            sort_by='accuracy',
        )
        vc_df

        # Plot confusion matrix for the best system
        best = max(vc_rows, key=lambda r: r.get('accuracy') or 0.0)
        sid = best['system_id']
        c = vc_counts[sid]
        tn, fp, fn, tp = int(c.get('tn') or 0), int(c.get('fp') or 0), int(c.get('fn') or 0), int(c.get('tp') or 0)
        if plt is not None:
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow([[tn, fp], [fn, tp]], cmap='Blues')
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Pred illegal', 'Pred legal'])
            ax.set_yticks([0, 1])
            ax.set_yticklabels(['True illegal', 'True legal'])
            ax.set_title(f"Validator confusion (best): {label_by_id.get(sid, sid)}")
            for (i, j, v) in [(0, 0, tn), (0, 1, fp), (1, 0, fn), (1, 1, tp)]:
                ax.text(j, i, str(v), ha='center', va='center', color='black')
            plt.show()
    else:
        print('No validator_baseline entries found in results.jsonl')

    # Generator parse errors
    ge_rows = generator_parse_errors(results_path)
    if ge_rows:
        show_table(
            [
                {
                    **r,
                    'system_label': label_by_id.get(r['system_id'], r['system_id']),
                }
                for r in ge_rows
            ],
            sort_by='parse_errors',
        )
    else:
        print('No generator parse errors found (or no generator entries in results.jsonl).')
